In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.cm as cm
import matplotlib.font_manager as fm
import numpy as np 
import seaborn as sns 
import math
import folium
import ipywidgets as widgets

import requests
import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go

### UNHCR - Forced displacement flow dataset

you can find the dataset here: 
https://www.unhcr.org/refugee-statistics/insights/explainers/forcibly-displaced-flow-data.html

In this project, we take a closer look at the United Nations Refugee Agency (UNHCR) dataset on Displacements. This dataset is a comprehensive list that records international desplacements from 1962 to 2025, where each entry has a country of "Origin", "Arrival" and a number of people. We have over 100.000 entries to look work with. Let's have a first look.

# 1 - Prepping the Data

In [ ]:
# load excel dataset
raw_data = pd.read_excel(
    '/Users/tish/Documents/Flexible Masters/02806 - Social data and visualization/CODE_Social_data_Visialization/TishDuc.github.io/Displacements_Final_Project/Data/UNHCR_Flow_Data.xlsx',
    sheet_name='DATA'
)

print(raw_data.shape)
raw_data[:10]

### 1.1 - Data Cleaning

Though the initial data looks rather clean to start with, we found a few duplicates, removed missung values and cleaned it a bit furthe for ease of use.

In [ ]:
# -------------------------------------------------------------------
#                      changes to the raw data 
# -------------------------------------------------------------------

#    lower casing
# ------------------
# header to lowercase
raw_data.columns = raw_data.columns.str.lower()
# data to lowercase
for col in raw_data.select_dtypes(include='object'):
    raw_data[col] = raw_data[col].str.lower()

# check for duplicates
# ---------------------
print("number of duplicates found:", raw_data.duplicated().sum())
# remove duplicates
raw_data = raw_data.drop_duplicates()

# check for missing values
# ---------------------
print(raw_data.isnull().sum()) 
# remove rows with missing values
raw_data = raw_data.dropna()


# -------------------------------------------------------------------

# Sanity check it worked:
# ---------------------
# print(raw_data.isnull().sum())
raw_data[:5]


In [ ]:
# ---------------------------------------------------------
#         Dealing with long impractical country names
# ---------------------------------------------------------

# change "originname"
name_edits = {
    "serbia and kosovo: s/res/1244 (1999)": "serbia/kosovo",
    "bolivia (plurinational state of)": "bolivia",
    "iran (islamic rep. of)": "iran",
    "netherlands (kingdom of the)": "nederland",
    "côte d'ivoire": "ivory coast",
    "state of palestine": "palestine",
    "lao people's dem. rep": "laos",
    "russian federation": "russia",
    "syrian arab rep.": "syria",
    "dem. rep. of the congo": "dr congo",
    "congo, republic of": "congo",
    "china, hong kong sar": "hong kong",
    "rep. of korea": "south korea",
    "dem. people's rep. of korea": "north korea",
    "central african rep.": "central african republic",
    "dominican rep.": "dominican republic",
    "united rep. of tanzania": "tanzania",
    "venezuela (bolivarian republic of)": "venezuela",
    "rep. of moldova": "moldova"
}

# count number of "serbia and kosovo: s/res/1244 (1999)" entries
print("number of 'serbia and kosovo: s/res/1244 (1999)' entries:", raw_data[raw_data['originname'] == "serbia and kosovo: s/res/1244 (1999)"].shape[0])
# count number of "côte d'ivoire" entries
print("number of 'côte d'ivoire' entries:", raw_data[raw_data['originname'] == "côte d'ivoire"].shape[0])
# coun t numbver of "iran (islamic rep. of)" entries
print("number of 'iran (islamic rep. of)' entries:", raw_data[raw_data['originname'] == "iran (islamic rep. of)"].shape[0])
print()


#            Apply changes
# -----------------------------------
# country name changes to the 'originname' column
raw_data['originname'] = raw_data['originname'].replace(name_edits)
raw_data["asylumname"] = raw_data["asylumname"].replace(name_edits)

# sanity check it worked
print("number of 'serbia/kosovo' entries after edit:", raw_data[raw_data['originname'] == "serbia/kosovo"].shape[0])
# number of "ivory coast" entries after edit
print("number of 'ivory coast' entries after edit:", raw_data[raw_data['originname'] == "ivory coast"].shape[0])
# number of "iran" entries after edit
print("number of 'iran' entries after edit:", raw_data[raw_data['originname'] == "iran"].shape[0])





### 1.2 - Exporting Cleaned Data

In [ ]:
# --------------------------------------------------------------
#             Export cleaned Data ----> CSV format
# --------------------------------------------------------------

clean_data = raw_data.copy()

# export cleaned data to csv
clean_data.to_csv('/Users/tish/Documents/Flexible Masters/02806 - Social data and visualization/CODE_Social_data_Visialization/TishDuc.github.io/Displacements_Final_Project/Data/cleaned_data.csv', index=False)

### 1.3 - Filtering 

Since the focus of our story, will be on major events in recent history we decided to apply a filter to the cleaned data, only to keep entries that recod 100 people or more. Initial analysis reveal that a great majority, i.e. over 76000 entries record "minor" numbers or displaced people. We filtered those out for our story but alsp kept the complete cleaned dataset for later use and comparaison.

In [ ]:
# -----------------------------------------------------------------
#                            filters 
# -----------------------------------------------------------------

#       Less than 100
# ----------------------------
# Count rows that have 'count' less than 100
less_than_100 = raw_data[raw_data['count'] < 100].shape[0]
print(f"Number of rows with 'count' less than 100: {less_than_100}")
# filter out rows where 'count' is less than 100
filtered_data = raw_data[raw_data['count'] >= 100].copy()


In [ ]:
# --------------------------------------------------------------
#            Export filtered Data ----> CSV format
# --------------------------------------------------------------

filtered_data = filtered_data.copy()
# export filtered data to csv
filtered_data.to_csv('/Users/tish/Documents/Flexible Masters/02806 - Social data and visualization/CODE_Social_data_Visialization/TishDuc.github.io/Displacements_Final_Project/Data/filtered_data.csv', index=False)

# 2 - A First Look 

Now that the data is cleaned, we can dive in and have a first look. 

How many datapoints do we have each year? is it a comparable amount? 

### 1.1 - Number of entries per year

In [ ]:
# ---------------------------------------------------------------
#                           data count 
# ---------------------------------------------------------------

# count number of entries per year for the cleaned data
num_entries_per_year_clean = clean_data['year'].value_counts().sort_index()
# print(num_entries_per_year_clean)

# count number of entries per year for filtered data
num_entries_per_year_filtered = filtered_data['year'].value_counts().sort_index()
# print(num_entries_per_year_filtered)

# common y-axis limit
max_count = max(
    num_entries_per_year_clean.max(),
    num_entries_per_year_filtered.max()
)

# plot the two histograms side by side
# -------------------------------------
plt.figure(figsize=(12, 5))

#           cleaned data
# -------------------------------------
plt.subplot(1, 2, 1)
plt.hist(
    clean_data['year'],
    bins=np.arange(
        clean_data['year'].min(),
        clean_data['year'].max() + 2
    ) - 0.5,
    color='darkcyan',
    alpha=0.6,
    edgecolor='lightgray',
    linewidth=0.5
)

plt.title('Number of Entries per Year (Cleaned Data)', pad=15, fontsize=12)
plt.xlabel('Year', labelpad=10)
plt.ylabel('Number of Entries', labelpad=10)
plt.xticks(rotation=45)
plt.ylim(0, max_count + 300)
plt.margins(y=0.05)
plt.grid(axis='x', alpha=0.2)
plt.grid(axis='y', alpha=0.2)


#           filtered data
# -------------------------------------
plt.subplot(1, 2, 2)
plt.hist(
    filtered_data['year'],
    bins=np.arange(
        filtered_data['year'].min(),
        filtered_data['year'].max() + 2
    ) - 0.5,
    color='orange',
    edgecolor='lightgray',
    linewidth=0.5
)

plt.title('Number of Entries per Year (Filtered Data)', pad=15, fontsize=12)
plt.xlabel('Year', labelpad=10)
plt.ylabel('Number of Entries', labelpad=10)
plt.xticks(rotation=45)
plt.ylim(0, max_count + 300)
plt.margins(y=0.05)
plt.grid(axis='x', alpha=0.2)
plt.grid(axis='y', alpha=0.2)

# change opacity of the frame
for ax in plt.gcf().axes:
    for spine in ax.spines.values():
        spine.set_edgecolor((0, 0, 0, 0.2))  # 20% opacity
        spine.set_linewidth(0.2)

plt.tight_layout()

plt.show()



Here's a closer look at entries between 1962 and 1981, since some of those values are relatively low

In [ ]:
# first 10 years
clean_first_20 = num_entries_per_year_clean.head(20)
filtered_first_20 = num_entries_per_year_filtered.head(20)

# create comparison table
comparison_table = pd.DataFrame({
    'Cleaned Data': clean_first_20,
    'Filtered Data': filtered_first_20
}).T

print(comparison_table)

## 1.2 - checking the filtering doesn't change the overall data

When filtering, we have removed over 76.000 entries of our initial data! this might seen alot, but it's good to keep inmind that those rows contained entries with count < 100. We wnated to be certain that removing 75% or the entries didn't change displacements counts nor the data's behavious too radically. 

So we plotted both: Displacement counts for the full "clean_data", and displacement counts for the "filtered_data"

In [ ]:
# -----------------------------------------
#           Displacements per year
# -----------------------------------------

# quick reminder:
# number of entries in each dataset
# ------------------------ 

# number of entries in clean_data
num_clean = clean_data.shape[0]
print("number of entries in clean_data:", num_clean)

# number of entries in filtered_data
num_filtered = filtered_data.shape[0]
print("number of entries in filtered_data:", num_filtered)
print()

#     Displacement count per year
# -------------------------------------

# with clean data
clean_data['count'] = pd.to_numeric(clean_data['count'], errors='coerce')
refugees_per_year = clean_data.groupby('year')['count'].sum().reset_index()
refugees_per_year = refugees_per_year.sort_values('year')

# with filtered data
filtered_data['count'] = pd.to_numeric(filtered_data['count'], errors='coerce')
refugees_per_year_filtered = filtered_data.groupby('year')['count'].sum().reset_index()
refugees_per_year_filtered = refugees_per_year_filtered.sort_values('year')

# compare
# ------------------
comparison_table = pd.DataFrame({
    'clean_data': refugees_per_year.set_index('year')['count'],
    'filtered_data': refugees_per_year_filtered.set_index('year')['count']
}).T

display(comparison_table.style.format("{:,.0f}"))

# Plotting the two
# ------------------

plt.figure(figsize=(12,6))
plt.plot(refugees_per_year["year"], refugees_per_year["count"], color='darkcyan', alpha=0.6, label='Cleaned Data')
plt.plot(refugees_per_year_filtered["year"], refugees_per_year_filtered["count"], color='orange', alpha=0.6, label='Filtered Data')
plt.xlabel("Year",fontsize=12, labelpad=10)
plt.ylabel("Forced displacement count", )
plt.title("Total displacements per year")
plt.legend()

# frame opacity
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_alpha(0.2)

plt.grid(True, alpha=0.3)
plt.show()



As we can clearly see, the two graphs almost overlap completely. Seing as removing 75% of the small entries makes such a minor difference in displacements counts, we will continue this project with the smaller "filtered data" and it's 30405 entries that represent the great majority of our data.

**Note on the term "Asylum seekers":**

This term only dates back from 1951, when it was implemented in the UN's Refugee Convention that same year. It is a status that seeks to protect refugees from being returned to countries at risk. It was later supplemented by the 1967 protocol, where the term was officially recognised worldwide. The first available data from teh UNCR dates back from 1962

**Back to our data:**

From our histograms, we immediately notice an increace in number of entries over time spaning from 10 entries in 1962, to over 4.500 displacements recorded for 2025. This could be due to several factors:
- An increase in sheer data collection over time, particularly after the year 2000, where we notice a sudden rise in numbers of entries
- More international movements in recent years? as opposed to internal displacements, which the dataset would not show.
- We are more people on the planet than we were in 1960 (3 billion in total in 1960 VS over 8 billion today) more people statistically means more total displacements.



### 1.2 - Countries and world regions

How many total countries are mentioned in this dataset? When are they first mentionned? 
Can we see some new counrty names appearing over time? for example: 
- Croatia, Estonia, Slovenia, Kazakhstan, Azerbaijan in the early 90's? 
- Eritrea in 1993? 
- Montenegro/Serbia 2006? 
- South Sudan in 2011? 

Are there countries that only "recieve" asylum seekers? 

Are there countries that only "send" asylunm seekers?

In [ ]:

#---------------------------------
#    unique ORIGIN countries
#---------------------------------
# total names and year
unique_origin_countries = clean_data['originname'].unique()
print("Total unique origin countries:", len(unique_origin_countries))

# year of the first and last mention of each country in the origin column
first_mention_origin_country = clean_data.groupby('originname')['year'].min()
last_mention_origin_country = clean_data.groupby('originname')['year'].max()

#---------------------------------
#   unique DESTINATION countries
#---------------------------------
# total name and year
unique_destination_countries = clean_data['asylumname'].unique()
print("Total unique destination countries:", len(unique_destination_countries))

# year of the first and last mention of each country in the destination column
first_mention_destination_country = clean_data.groupby('asylumname')['year'].min()
last_mention_destination_country = clean_data.groupby('asylumname')['year'].max()

#---------------------------------
#  COMBINED: unique country names
#---------------------------------
unique_countries = set(unique_origin_countries).union(set(unique_destination_countries))
print("Total unique countries (origin + destination):", len(unique_countries))
print()
# print first 10 unique countries (origin + destination)
# print("frist ten unique countries:", list(unique_countries)[:10]) 
# find the 3 countries that are only in the destination column and not in the origin column
only_destination_countries = set(unique_destination_countries) - set(unique_origin_countries)
print("Countries that are only in the destination column", only_destination_countries)
print()

# ---------------------------------
#          World Regions
# ---------------------------------
# region of each unique countries
region_unique_countries = clean_data.groupby('asylumname')['asylumregion'].first().to_dict()


#---------------------------------
#        COMBINED dataframe
#---------------------------------
first_mention_country = pd.concat([
    first_mention_origin_country,
    first_mention_destination_country
], axis=1).min(axis=1)

last_mention_country = pd.concat([
    last_mention_origin_country,
    last_mention_destination_country
], axis=1).max(axis=1)

# combine first and last year into one dataframe
country_years = pd.DataFrame({
    'country': first_mention_country.index,
    'start': first_mention_country.values,
    'end': last_mention_country.values
})


#---------------------------------
#        regroup per region
#---------------------------------

# add region column to the country_years dataframe
country_years['region'] = country_years['country'].map(region_unique_countries)
# number or regions
print("Number of unique regions:", country_years['region'].nunique())
print("Unique regions:", country_years['region'].unique())
print()
# find input with Nan
print("Countries with missing region:", country_years[country_years['region'].isna()]['country'].tolist())
# sort by regions
country_years = country_years.sort_values(by='region')


country_years_by_region = {
    region: df.reset_index(drop=True)
    for region, df in country_years.groupby('region')
}

region_colors = {
    'europe': 'deepskyblue',
    'middle east and north africa': 'crimson',
    'asia and the pacific': 'limegreen',
    'eastern and southern africa': 'orange',
    'americas': 'deeppink',
    'west and central africa': 'darkcyan'
}


In [ ]:
#-------------------------------------------------------------
#            Plotting first and last appearance of
#        countries in the dataset, grouped by region
#-------------------------------------------------------------

# loop over regions
for region, df_region in country_years.groupby('region'):

    df_region = df_region.sort_values(by='country').reset_index(drop=True)

    n = len(df_region)
    plt.figure(figsize=(12, n * 0.25))

    for i, row in df_region.iterrows():

        # thin background line
        plt.hlines(
            y=i,
            xmin=country_years['start'].min(),
            xmax=country_years['end'].max(),
            color='gray',
            linewidth=0.3,
            alpha=0.8
        )

        plt.plot(
            [row['start'], row['end']],
            [i, i],
            region_colors.get(region, 'grey'),
            alpha=0.6,
            linewidth=3
        )

    plt.yticks(range(len(df_region)), df_region['country'], fontsize=9)
    plt.xlabel('Year')
    plt.ylabel('Country')
    plt.title(f'First and Last Appearance of Countries — {region}', pad=25)

    # frame opacity
    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_alpha(0.2)

    plt.margins(x=0, y=0.05)
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()

    plt.show()

These new barplots show the first and last time a country was mentionned in our dataset. 

coincide with what we saw earlier, there is less data initially between 1962 and the 80's. We see countres slowly appearing in our dataset Spain and France (1970), Denmark (1972), 


- first mentions are "first mentions" they don't always mean country creation Erithea 1994 mentionned earlier, demnark (date), looks like the name are retroactively changed. No trace of "yougoslavie" for ex.
- creation of recent countries:  european countries that appear in 1991 and in the middle east too + montenegro + South sudan etc...


### 1.3 Types of status

In our dataset, there are four status types that are recorded:

- **Asylum seekers (ASY):** people who have applied for asylum and are waiting for a decision on their application

- **Refugees (REF):** People who have been forced to flee their country because of persecution, war, or violence and have been recognized as refugees by the UNHCR. More precisely "defined by the 1951 Geneva Convention, any person who, owing to a well‐founded fear of persecution for reasons of race, religion, nationality, membership of a particular social group or political opinion, is outside the country of his or her nationality...and is  unable or, owing to such fear, is unwilling to return to it”. (ref: https://www.unesco.org/en/articles/migrants-refugees-or-displaced-persons)

- **Other people in need of international protection” (OIP):** defined by the UNHCR as “people who are outside their country or territory of origin, typically because they have been forcibly displaced across international borders, who have not been reported under other categories (asylum-seekers, refugees, people in refugee-like situations) but who likely need international protection. (ref: https://data.unicef.org/topic/child-migration-and-displacement/displacement/)

- **Refugees and others of concern":** A status that is less universally standardized, but in migration/displacement databases it often refered.


We now revisit the graph of "total displacements per year", this time separating the data by displacement status.


In [ ]:
# ----------------------
#     People status
# ----------------------
# plot total displaced people per year, per category
pt_per_year = filtered_data.groupby(['year', 'population types'])['count'].sum().reset_index()
pt_per_year = pt_per_year.sort_values('year')

plt.figure(figsize=(12,6))
for pt in pt_per_year['population types'].unique():
    subset = pt_per_year[pt_per_year['population types'] == pt]
    plt.plot(subset['year'], subset['count'], label=pt.upper())
plt.xlabel('Year',fontsize=12, labelpad=10)
plt.ylabel('Forced displacement count', labelpad=10)
plt.title('Total displacements per year, by population types', pad=15, fontsize=12)
plt.grid(True, alpha=0.3)

# change opacity of the frame
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_alpha(0.2)

plt.grid(True, alpha=0.3)
plt.legend()


# 3 - Displacements in recent History

After this initial look at the data, we dive in to the heard of the project: the displacements.



### 3.1 Finding large displacements, finding "crisis"

#### 3.1.1 - Sheer number of displacements

Here is what we took into consideration.
-  Conflicts/crisis vary greatly in length for example Rwanda genocide (about half a year), Syrian war (over 15 years). One million people moving in one year will not have the same effect as one million people moving over the course of 15 years. So we decided to track "crisis" based on sudden in creases in displacements in our data.
- we start tracking a conflict after detecting an entry of 10.000 OR after detecting the country's median yearly displacements

In [ ]:
# -------------------------------------------------
#            Finding the largest crisis
# -------------------------------------------------
# episode-based conflict significance analysis (variable-length conflicts)

#    parameters
# -----------------
min_activation = 10000          # threshold: minimum yearly displaced for a "crisis"    
activation_multiplier = 1.0     # max(10000, country_median_displacement)
quiet_years_to_end = 1          # end episode after 1 year below threshold         
duration_weight_alpha = 0.25    # long conflicts mathematically weighted up    

# sum across all asylum countries
# ------------------------------------
origin_yearly = (
    filtered_data.groupby(['origin', 'originname', 'year'], as_index=False)['count']
    .sum()
)

# ------------------------------------
#     Search for crisis in data  
# ------------------------------------
# runs separately for each origin country
def detect_origin_episodes(group):
    # create a time series of yearly displaced for the origin country
    series = group.set_index('year')['count'].sort_index() #
    full_years = range(int(series.index.min()), int(series.index.max()) + 1) 
    series = series.reindex(full_years, fill_value=0)

    #    "crisis" threshold
    # --------------------------
    positive = series[series > 0]
    # median-based threshold for detecting the episode  
    baseline = float(positive.median()) if not positive.empty else 0.0 #
    # final threshold is the max between median-based and 10000
    threshold = max(min_activation, baseline * activation_multiplier)

    # store detected episodes
    episodes = [] #
    in_episode = False 
    start_year = None
    low_streak = 0 # set to 0 when in episode, counts consecutive years below threshold

    #    scan year by year
    # --------------------------
    for year, value in series.items():
        # add new episode
        if not in_episode and value >= threshold:
            in_episode = True
            start_year = int(year)
            low_streak = 0
            continue
        #  update status of existing episode
        if in_episode:
            # if value falls below threshold, count it as a "low year"
            if value < threshold:
                low_streak += 1
                if low_streak >= quiet_years_to_end:
                    end_year = int(year) - quiet_years_to_end
                    if end_year >= start_year:
                        episodes.append((start_year, end_year, threshold))
                    in_episode = False
                    start_year = None
                    low_streak = 0
            # if the value is above threshold continue
            else:
                low_streak = 0

    # acontinue an episode that is still active
    if in_episode and start_year is not None:
        episodes.append((start_year, int(series.index.max()), threshold))

    # store episode details for this origin country
    rows = []

    # compute episode metrics
    # --------------------------
    for episode_num, (start, end, threshold_used) in enumerate(episodes, start=1):
        episode_series = series.loc[start:end]
        total_displaced = float(episode_series.sum())
        peak_annual = float(episode_series.max())
        duration_years = int(end - start + 1)
        mean_annual = float(total_displaced / duration_years)
        ramp_up = float(episode_series.diff().fillna(0).max())

        rows.append({
            'origin': group['origin'].iloc[0],
            'originname': group['originname'].iloc[0],
            'episode': episode_num, #refers to the number of forced displacements detected in that origin country
            'start_year': start,
            'end_year': end,
            'duration_years': duration_years,
            'threshold_used': float(threshold_used), # median-based threshold for detecting the episode
            'total_displaced': total_displaced,
            'peak_annual': peak_annual,
            'mean_annual': mean_annual,
            'ramp_up': ramp_up, # how quickly the episode escalated in its first year (max year-over-year increase)
        })

    # returns metrics for all episodes detected in that origin country
    return rows

#  run episode detection 
# for each origin country
# --------------------------
episode_rows = []
for _, g in origin_yearly.groupby(['origin', 'originname']):
    episode_rows.extend(detect_origin_episodes(g))
# create dataframe 
episodes_df = pd.DataFrame(episode_rows)

# --------------------------
#  compute composite score 
# --------------------------
# (total displaced weighted by duration)
episodes_df['composite_score'] = episodes_df['total_displaced'] * (
    1 + duration_weight_alpha * np.log1p(episodes_df['duration_years'])
)

# rank episodes
# ----------------
# ranked by total displaced in the whole duration of the episode
top_by_total = episodes_df.sort_values('total_displaced', ascending=False).reset_index(drop=True).head(12) 
print('Top episodes by total displaced:')
display(top_by_total)


In [ ]:
# ---------------------------------------------------------------
#     plotting the 10 largest displacements by total displaced
# ---------------------------------------------------------------

if episodes_df.empty:
    print("No episodes detected based on the defined criteria.")
else:
    # prepare data for plotting
    chart_data = top_by_total.copy()
    chart_data['episode_label'] = (
        chart_data['originname'].str.title() + ' (' +
        chart_data['start_year'].astype(str) + '-' +
        chart_data['end_year'].astype(str) + ')'
    )

    #       plot:
    # -------------------
    plt.figure(figsize=(12, 6))
    max_value = chart_data['total_displaced'].max()

    # background bars
    plt.barh(
        chart_data['episode_label'],
        [max_value] * len(chart_data),
        color='darkslategray',
        alpha=0.08
    )

    # actual data bars
    plt.barh(chart_data['episode_label'], chart_data['total_displaced'], color="darkcyan", alpha=0.5, )
    plt.xlabel('Total Displaced (in millions of people)', fontsize=10, labelpad=10)
    plt.ylabel('Episode',fontsize=10, labelpad=10)
    plt.title('Top 10 Episodes by Total Displaced', fontsize=12, pad=15)
    plt.gca().invert_yaxis()  # largest on top
    
    # frame opacity
    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_alpha(0.2)

    plt.margins(x=0.02, y=0.05)
    plt.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()


 !!!!!!!!!!!!!!!!!!!!!!!!!! NOTE mention the outlier situation of Afghanistan !!!!!!!!!!!!!!!!!!!!!!!!!!

#### 3.1.2 Sorted by other filters

We looked at ranking the episodes by "Annual Peaks" of displacements, as well as created a composite score that takes into consideration the length of conflicts and ranks them higher up.

In [ ]:
#  ranked by peak annual displacements
# -------------------------------------
top_by_peak = episodes_df.sort_values('peak_annual', ascending=False).reset_index(drop=True).head(12) 
# ranked by peak annual displacements
print('Top episodes by peak annual displaced:')
display(top_by_peak)



conclusions here?
- cheer numbers VS composite score?
- Syria is 1st (ni both)
- Venezuela 2nd (in both)
- Ukraine is 3rd (in both)

- "unknown/Other" = or missing data? ex-Yougoshavia and/or regions that changed name during the fall of the USSR? or people not wanting to reveal their origins?
- we used the filtered data (what would it look like with cleaned unfiltered data)?

# 4 - Mapping our research

In [ ]:
# -------------------------------
#       prepping for mapping 
# -------------------------------

resp = requests.get("https://restcountries.com/v3.1/all?fields=cca3,latlng")
country_coords = {}
for c in resp.json():
    iso3 = c.get("cca3", "")
    latlng = c.get("latlng", [])
    if iso3 and len(latlng) == 2:
        country_coords[iso3] = (float(latlng[0]), float(latlng[1]))  # (lat, lon)

print(f"Loaded centroids for {len(country_coords)} countries")
print(f"Sample keys: {list(country_coords.keys())[:6]}")

# Use filtered_data (lowercase columns)
all_iso = set(filtered_data['originiso'].dropna().str.upper()) | set(filtered_data['asylumiso'].dropna().str.upper())
matched = all_iso & set(country_coords.keys())
missing = all_iso - set(country_coords.keys())
print(f"ISO codes in dataset: {len(all_iso)}, matched: {len(matched)}, missing: {len(missing)}")
print(f"Missing sample: {list(missing)[:10]}")




In [ ]:
# ----------------------------------------------------------------------------
#                                    MAP 1
# ----------------------------------------------------------------------------

# -------------------------------
#          flow counts 
# -------------------------------


# count total displaced per flow (origin-destination-year)
# -------------------------------
flow_df = (
    filtered_data.groupby(['year', 'originiso', 'asylumiso'], as_index=False)['count'].sum()
)
# only keep top 30 flows per year for mapping
# TOP_N = 30 
TOP_N = 100
# apply filter
top_flows = (
    flow_df.sort_values('count', ascending=False)
    .groupby('year', group_keys=False)
    .head(TOP_N)
    .reset_index(drop=True)
)

print(f"Flow records: {len(top_flows)} across {top_flows['year'].nunique()} years")



# Enrich origin_year dataset
# -------------------------------
# count total displaced people per origin country and year
origin_year = (
    filtered_data.groupby(['year', 'originiso', 'originname'], as_index=False)['count'].sum()
)

# Count number of unique destination countries
# reached by each origin country per year
dest_counts = (
    filtered_data.groupby(['year', 'originiso'])['asylumiso']
    .nunique()
    .reset_index(name='n_dest')
)
# Merge displacement totals with destination counts
origin_enriched = origin_year.merge(dest_counts, on=['year', 'originiso'], how='left')
# Replace missing values and convert to integer
origin_enriched['n_dest'] = origin_enriched['n_dest'].fillna(0).astype(int)

# ---------------------------------------------------------------------------------------
# ---------------------------------------------------------------------------------------


# Animation + visualisation settings
# -----------------------------------
# List of years used for animation frames
years_list = sorted(origin_enriched['year'].unique())
# Number of line-width buckets for flows
N_BUCKETS  = 5
# Total number of traces:
# 1 choropleth + flow buckets + arrow buckets
N_TRACES   = 1 + N_BUCKETS + N_BUCKETS + 1


# Global logarithmic colour scaling
# -----------------------------------
# Fix colour scale maximum to 1 million
# so colours remain consistent across years
global_max_c = 4_000_000
# Log-scaled upper bound
log_max = float(np.log1p(global_max_c))
# Minimum threshold for colour intensity
# values below 1k appear very light
log_min = float(np.log1p(1_000))
# Maximum flow size used for line scaling
global_max_f = float(np.log1p(top_flows['count'].max())) if not top_flows.empty else 1.0
global_log_max_f = float(global_max_f if global_max_f > 0 else 1.0)


#         colourbar ticks
# -----------------------------------
_tick_raw  = [1_000, 5_000, 10_000, 50_000, 100_000, 500_000, 1_000_000, 4_000_000]
# Convert tick values to logarithmic scale
_tick_vals = [float(np.log1p(v)) for v in _tick_raw if v <= global_max_c]
# Labels shown on colourbar
_tick_text = ["1k", "5k", "10k", "50k", "100k", "500k", "1M", "4M"][:len(_tick_vals)]



#           Flow styling
# ----------------------------------

# hex → rgba
def hex_to_rgba(hex_color, alpha):
    r, g, b = mcolors.to_rgb(hex_color)
    return f"rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {alpha})"

# flow colour from ratio
def flow_color_from_ratio(ratio, alpha=0.60): # ------------------------------ alpha is the opacity of the flow color
    """
    Convert normalized flow intensity into RGBA colour.
    Larger flows become darker.
    """
    custom_colors = [
        "#e0e0e0",
        "#bdbdbd",
        "#878787",
        "#646464",
        "#383838",
        "#000000",
    ]

    cmap = LinearSegmentedColormap.from_list("custom_greys", custom_colors)

    r, g, b, _ = cmap(ratio)
    return f"rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {alpha})"

# shared color picker
FLOW_COLOR_PICKER = widgets.ColorPicker(
    value="#444444",
    description='Flows'
)

# shared flow color + opacity
FLOW_COLOR = hex_to_rgba(FLOW_COLOR_PICKER.value, 1)

# line thickness buckets
BUCKET_WIDTHS = [1.5, 1.5, 1.5, 1.5, 1.5] # fxed size afterall
# try with 1.2
# BUCKET_WIDTHS = [1.2, 1.2, 1.2, 1.2, 1.2] # fxed size afterall
# try with 1
# BUCKET_WIDTHS = [1, 1, 1, 1, 1] # fxed size afterall

# flow colour buckets
BUCKET_COLORS = [
    flow_color_from_ratio((i + 1) / N_BUCKETS)
    for i in range(N_BUCKETS)
]



# Create curved geographic flow arcs
# -----------------------------------
def curved_arc(lat1, lon1, lat2, lon2, n=50, height=0.12):
    """
    Create a curved arc between two geographic points.
    Parameters
    ----------
    lat1, lon1 : float
        Origin coordinates
    lat2, lon2 : float
        Destination coordinates
    n : int
        Number of interpolation points
    height : float
        Arc curvature intensity
    """

    t = np.linspace(0, 1, n)

    # Linear interpolation between points
    lats = lat1 + t * (lat2 - lat1)
    lons = lon1 + t * (lon2 - lon1)

    # Direction vector
    dlat, dlon = lat2 - lat1, lon2 - lon1
    dist = (dlat**2 + dlon**2) ** 0.5
    # Avoid instability for very short distances
    if dist < 0.01:
        return lats.tolist(), lons.tolist()
    # Perpendicular vector to create curvature
    perp_lat, perp_lon = -dlon / dist, dlat / dist
    # Bell-shaped bulge along the curve
    bulge = 4 * t * (1 - t) * height * dist
    return (
        lats + bulge * perp_lat
    ).tolist(), (
        lons + bulge * perp_lon
    ).tolist()



#    Compute arrow orientation
# -----------------------------------
def calc_bearing(lat1, lon1, lat2, lon2):

    # Convert to radians
    r = math.radians
    rl1, rl2 = r(lat1), r(lat2)
    dlon = r(lon2 - lon1)

    # Bearing formula
    x = math.sin(dlon) * math.cos(rl2)
    y = (
        math.cos(rl1) * math.sin(rl2)
        - math.sin(rl1) * math.cos(rl2) * math.cos(dlon)
    )
    return (math.degrees(math.atan2(x, y)) + 360) % 360


# -----------------------------------------------------------------
#                Build traces for a single year
# -----------------------------------------------------------------

def make_traces(year):

    # Data for selected year
    yd = origin_enriched[origin_enriched['year'] == year].copy()
    yf = top_flows[top_flows['year'] == year]

    # Choropleth layer
    # -------------------
    # Custom hover data
    cd = np.column_stack([
        yd['originname'].fillna(yd['originiso'].str.upper()).values,
        [f"{int(x):,}" for x in yd['count'].values],
        yd['n_dest'].values.astype(str),
    ])

    choropleth = go.Choropleth(
        # Country ISO codes
        locations=yd['originiso'].str.upper(),
        # Colour intensity
        z=np.log1p(yd['count'].values),
        # Shared colour scale
        colorscale='Reds',
        zmin=log_min,
        zmax=log_max,
        # Hover information
        customdata=cd,

        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "────────────────────<br>"
            "People displaced: <b>%{customdata[1]}</b><br>"
            "Destination countries: <b>%{customdata[2]}</b>"
            "<extra></extra>"
        ),

        colorbar=dict(
            title=dict(
                text="People leaving<br>per country<br>(log scale)<br>...........",
                font=dict(size=12)
            ),
            tickvals=_tick_vals,
            ticktext=_tick_text,
            tickfont=dict(size=14),
            x=1.01,
            thickness=14,
        ),

        showscale=True,

        # Country borders
        marker_line_width=0.3,
        marker_line_color='white',
    )

    # Curved flow arcs + arrowheads
    # -----------------------------------
    # Containers for each flow-width bucket
    bucket_lons = [[] for _ in range(N_BUCKETS)]
    bucket_lats = [[] for _ in range(N_BUCKETS)]

    # Arrowhead containers
    a_lons = [[] for _ in range(N_BUCKETS)]
    a_lats = [[] for _ in range(N_BUCKETS)]
    a_angles = [[] for _ in range(N_BUCKETS)]
    a_sizes = [[] for _ in range(N_BUCKETS)]
    a_text = [[] for _ in range(N_BUCKETS)]

    for _, row in yf.iterrows():

        # Origin and destination coordinates
        o = country_coords.get(str(row['originiso']).upper())
        a = country_coords.get(str(row['asylumiso']).upper())

        # Skip missing coordinates
        if o is None or a is None:
            continue

        # Relative flow intensity
        ratio = min(
            np.log1p(row['count']) / global_log_max_f,
            1.0
        )

        # Width bucket assignment
        bucket = min(
            int(
                max(
                    0,
                    min(
                        (
                            (np.log1p(row['count']) - np.log1p(500))
                            / (np.log1p(1_000_000) - np.log1p(500))
                        ),
                        1
                    )
                ) * N_BUCKETS
            ),
            N_BUCKETS - 1
        )


        # Generate curved path
        arc_lats, arc_lons = curved_arc(
            o[0], o[1],
            a[0], a[1]
        )

        # Store arc coordinates
        bucket_lons[bucket] += arc_lons + [None]
        bucket_lats[bucket] += arc_lats + [None]

        # Arrowhead position
        a_lons[bucket].append(a[1])
        a_lats[bucket].append(a[0])

        # Arrow orientation
        if len(arc_lats) >= 2 and len(arc_lons) >= 2:
            a_angles[bucket].append(
                calc_bearing(
                    arc_lats[-2],
                    arc_lons[-2],
                    arc_lats[-1],
                    arc_lons[-1]
                )
            )
        else:
            a_angles[bucket].append(0)

        # Arrow size proportional to flow
        # -----------------------------
        # make smaller arrows
        # a_sizes.append(4 + ratio * 8)
        # ------------------------------------------------------------------------------
        a_sizes[bucket].append(2 + ratio * 5) # minimum size of 3, maximum size of 7


        # Hover label
        a_text[bucket].append(
            f"<b>{str(row['originiso']).upper()} → {str(row['asylumiso']).upper()}</b><br>"
            f"─────────────────<br>"
            f"People moving: <b>{int(row['count']):,}</b>"
        )

    #           Line traces
    # -----------------------------------
    line_traces = []
    for b in range(N_BUCKETS):
        line_traces.append(go.Scattergeo(
            geo='geo',
            lon=bucket_lons[b] or [None],
            lat=bucket_lats[b] or [None],
            mode='lines',
            line=dict(
                width=BUCKET_WIDTHS[b],
                color=BUCKET_COLORS[b]
            ),
            showlegend=False,
            hoverinfo='skip',
        ))

    #       Arrowhead layer
    # ------------------------------------------------------------------------------
    arrow_traces = []
    for b in range(N_BUCKETS):
        arrow_traces.append(go.Scattergeo(
            geo='geo',
            lon=a_lons[b] or [None],
            lat=a_lats[b] or [None],
            mode='markers',
            marker=dict(
                symbol='arrow-up',
                size=a_sizes[b] or [0],
                color=BUCKET_COLORS[b],
                angle=a_angles[b] or [0],
                line=dict(width=0),
            ),
            text=a_text[b] or [''],
            hovertemplate='%{text}<extra></extra>',
            showlegend=False,
        ))

    #       Flow colourbar
    # ------------------------------------------------------------------------------
    flow_colorbar = go.Scattergeo(
        geo='geo',
        lon=[0],
        lat=[0],
        mode='markers',
        marker=dict(
            size=0.01,
            color=[np.log1p(500), np.log1p(1_000_000)],
            #colorscale='Greys',
            # custom colors
            colorscale=[
                [0.00, "#dcdcdc"],
                [0.15, "#bdbdbd"],
                [0.35, "#878787"],
                [0.55, "#646464"],
                [0.75, "#383838"],
                [1.00, "#000000"],
            ],
            cmin=np.log1p(500),
            cmax=np.log1p(1_000_000),
            showscale=True,
            colorbar=dict(
                title=dict(
                    text="Flow of <br>people<br>moving<br>(log scale)<br>...........",
                    font=dict(size=12),
                    side="top"
                ),
            tickvals=[
                np.log1p(500),
                np.log1p(1_000),
                np.log1p(5_000),
                np.log1p(10_000),
                np.log1p(100_000),
                np.log1p(500_000),
                np.log1p(1_000_000),
            ],

            ticktext=["500", "1k", "5k", "10k", "100k", "500k", "1M"],
                x=-0.08,
                thickness=14,
            ),
        ),
        hoverinfo='skip',
        showlegend=False,
    )

    return [choropleth] + line_traces + arrow_traces + [flow_colorbar]


# -----------------------------------
#           Initial figure
# -----------------------------------
fig = go.Figure(data=make_traces(years_list[0]))

fig.frames = [
    go.Frame(
        data=make_traces(y),
        traces=list(range(N_TRACES)),
        name=str(y)
    )
    for y in years_list
]

#       Layout control
# ----------------------------
fig.update_layout(

    title=dict(
        text='Top 100 Largest displacements worldwide per year',
        x=0.5,
        xanchor='center',
        yanchor='top',
        font=dict(color="#383838", family='helvetica', size=16)
    ),

    geo=dict(
        showframe=False,

        showcoastlines=True,
        coastlinecolor='lightgray',

        showcountries=True,
        countrycolor="#C0E3FF",
        countrywidth=0.9,

        showland=True,
        landcolor="#efefef",

        showocean=True,
        oceancolor="#C0E3FF",

        bgcolor="#E8E8E8",
        projection_type='natural earth',
    ),
    

    # Play / pause toggle button
    updatemenus=[{
            "type": "buttons", "showactive": False,
            "y": 0.02, "x": 0.05, "xanchor": "right",
            "buttons": [
                {"label": "\u25b6 Play", "method": "animate",
                "args": [None, {"frame": {"duration": 1800}, "fromcurrent": True,
                                "transition": {"duration": 400}}]},
                {"label": "\u23f8 Pause", "method": "animate",
                "args": [[None], {"frame": {"duration": 0}, "mode": "immediate"}]},
            ],
        }],
        sliders=[{
            "active": 0,
            "steps": [
                {"args": [[str(y)], {"frame": {"duration": 1800}, "mode": "immediate",
                                    "transition": {"duration": 300}}],
                "label": str(y), "method": "animate"}
                for y in years_list
            ],
            "x": 0.1, "len": 0.85, "y": 0,
            "currentvalue": {"prefix": "Year: ", "font": {"size": 12}, "xanchor": "center"},
            "pad": {"b": 10, "t": 50},
        }],
        margin=dict(l=80, r=80, t=50, b=80),
        height=600,
    )

#            Export & show
# -----------------------------------

fig.write_html("../visualizations/Top_100_displacements_flow_map.html")

fig.show()

notes on this section ? 
we encourage the reager to explore this map for themselves and see how tracking refugees flows reveals crisis of the modern era.

NOTE : the map does not take into consideration new countries, nor change of fronteers over the years.  

what we notice
- again more data over the years
- Do people have a tendency to "favor" neighboring countries when escaping a crisis? 
- Do people seem to travel longer and longer distances over the years? As the decades pass and it gets easier and more accessible to fly accross the world?
- Any parts of the worls seem particularly attractive? where doe people go? From our mapped data Europem USA, Canada seems to be a popular place.




In [ ]:
# ----------------------------------------------------------------------------
#                                    MAP 2
# ----------------------------------------------------------------------------

# -------------------------------
#          flow counts 
# -------------------------------


# count total displaced per flow (origin-destination-year)
# -------------------------------
flow_df = (
    filtered_data.groupby(['year', 'originiso', 'asylumiso'], as_index=False)['count'].sum()
)
# only keep top 30 flows per year for mapping
# TOP_N = 30 
TOP_N = 100
# apply filter
top_flows = (
    flow_df.sort_values('count', ascending=False)
    .groupby('year', group_keys=False)
    .head(TOP_N)
    .reset_index(drop=True)
)

print(f"Flow records: {len(top_flows)} across {top_flows['year'].nunique()} years")



# Enrich origin_year dataset
# -------------------------------
# count total displaced people per origin country and year
origin_year = (
    filtered_data.groupby(['year', 'originiso', 'originname'], as_index=False)['count'].sum()
)

# Count number of unique destination countries
# reached by each origin country per year
dest_counts = (
    filtered_data.groupby(['year', 'originiso'])['asylumiso']
    .nunique()
    .reset_index(name='n_dest')
)
# Merge displacement totals with destination counts
origin_enriched = origin_year.merge(dest_counts, on=['year', 'originiso'], how='left')
# Replace missing values and convert to integer
origin_enriched['n_dest'] = origin_enriched['n_dest'].fillna(0).astype(int)

# ---------------------------------------------------------------------------------------
# ---------------------------------------------------------------------------------------


# Animation + visualisation settings
# -----------------------------------
# List of years used for animation frames
years_list = sorted(origin_enriched['year'].unique())
# Number of line-width buckets for flows
N_BUCKETS  = 5
# Total number of traces:
# 1 choropleth + flow buckets + arrow buckets
N_TRACES   = 1 + N_BUCKETS + N_BUCKETS + 1


# Global logarithmic colour scaling
# -----------------------------------
# Fix colour scale maximum to 1 million
# so colours remain consistent across years
global_max_c = 4_000_000
# Log-scaled upper bound
log_max = float(np.log1p(global_max_c))
# Minimum threshold for colour intensity
# values below 1k appear very light
log_min = float(np.log1p(1_000))
# Maximum flow size used for line scaling
global_max_f = float(np.log1p(top_flows['count'].max())) if not top_flows.empty else 1.0
global_log_max_f = float(global_max_f if global_max_f > 0 else 1.0)


#         colourbar ticks
# -----------------------------------
_tick_raw  = [1_000, 5_000, 10_000, 50_000, 100_000, 500_000, 1_000_000, 4_000_000]
# Convert tick values to logarithmic scale
_tick_vals = [float(np.log1p(v)) for v in _tick_raw if v <= global_max_c]
# Labels shown on colourbar
_tick_text = ["1k", "5k", "10k", "50k", "100k", "500k", "1M", "4M"][:len(_tick_vals)]



#           Flow styling
# ----------------------------------

# hex → rgba
def hex_to_rgba(hex_color, alpha):
    r, g, b = mcolors.to_rgb(hex_color)
    return f"rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {alpha})"

# flow ratio → rgba
def flow_color_from_ratio(ratio, alpha=0.80): # ------------------------------ alpha is the opacity of the flow color
    """
    Convert normalized flow intensity into RGBA colour.
    Larger flows become darker.
    """
    # cmap = cm.get_cmap("Purples")
    # custome colors
    custom_purples = LinearSegmentedColormap.from_list(
        "custom_purples",
        [
            "#e9d6fc",
            "#d9b3ff",
            "#b266ff",
            "#8c1aff",
            "#42006e",
            "#14001f",
        ]
    )
    cmap = custom_purples

    r, g, b, _ = cmap(ratio)
    return f"rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {alpha})"

# shared color picker
FLOW_COLOR_PICKER = widgets.ColorPicker(
    value="#444444",
    description='Flows'
)

# shared flow color + opacity
FLOW_COLOR = hex_to_rgba(FLOW_COLOR_PICKER.value, 1)

# line thickness buckets
# BUCKET_WIDTHS = [1.5, 1.5, 1.5, 1.5, 1.5] # fxed size afterall
# try with 1.2
BUCKET_WIDTHS = [1.2, 1.2, 1.2, 1.2, 1.2] # fxed size afterall
# try with 1
# BUCKET_WIDTHS = [1, 1, 1, 1, 1] # fxed size afterall

# flow colour buckets
BUCKET_COLORS = [
    flow_color_from_ratio(0.25 + 0.95 * (i / (N_BUCKETS - 1)))
    for i in range(N_BUCKETS)
]



# Create curved geographic flow arcs
# -----------------------------------
def curved_arc(lat1, lon1, lat2, lon2, n=50, height=0.12):
    """
    Create a curved arc between two geographic points.
    Parameters
    ----------
    lat1, lon1 : float
        Origin coordinates
    lat2, lon2 : float
        Destination coordinates
    n : int
        Number of interpolation points
    height : float
        Arc curvature intensity
    """

    t = np.linspace(0, 1, n)

    # Linear interpolation between points
    lats = lat1 + t * (lat2 - lat1)
    lons = lon1 + t * (lon2 - lon1)

    # Direction vector
    dlat, dlon = lat2 - lat1, lon2 - lon1
    dist = (dlat**2 + dlon**2) ** 0.5
    # Avoid instability for very short distances
    if dist < 0.01:
        return lats.tolist(), lons.tolist()
    # Perpendicular vector to create curvature
    perp_lat, perp_lon = -dlon / dist, dlat / dist
    # Bell-shaped bulge along the curve
    bulge = 4 * t * (1 - t) * height * dist
    return (
        lats + bulge * perp_lat
    ).tolist(), (
        lons + bulge * perp_lon
    ).tolist()



#    Compute arrow orientation
# -----------------------------------
def calc_bearing(lat1, lon1, lat2, lon2):

    # Convert to radians
    r = math.radians
    rl1, rl2 = r(lat1), r(lat2)
    dlon = r(lon2 - lon1)

    # Bearing formula
    x = math.sin(dlon) * math.cos(rl2)
    y = (
        math.cos(rl1) * math.sin(rl2)
        - math.sin(rl1) * math.cos(rl2) * math.cos(dlon)
    )
    return (math.degrees(math.atan2(x, y)) + 360) % 360


# -----------------------------------------------------------------
#                Build traces for a single year
# -----------------------------------------------------------------

def make_traces(year):

    # Data for selected year
    yd = origin_enriched[origin_enriched['year'] == year].copy()
    yf = top_flows[top_flows['year'] == year]

    # Choropleth layer
    # -------------------
    # Custom hover data
    cd = np.column_stack([
        yd['originname'].fillna(yd['originiso'].str.upper()).values,
        [f"{int(x):,}" for x in yd['count'].values],
        yd['n_dest'].values.astype(str),
    ])

    choropleth = go.Choropleth(
        # Country ISO codes
        locations=yd['originiso'].str.upper(),
        # Colour intensity
        z=np.log1p(yd['count'].values),
        # Shared colour scale
        colorscale='Reds',
        zmin=log_min,
        zmax=log_max,
        # Hover information
        customdata=cd,

        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "────────────────────<br>"
            "People displaced: <b>%{customdata[1]}</b><br>"
            "Destination countries: <b>%{customdata[2]}</b>"
            "<extra></extra>"
        ),

        colorbar=dict(
            title=dict(
                text="People leaving<br>per country<br>(log scale)<br>..........",
                font=dict(size=12)
            ),
            tickvals=_tick_vals,
            ticktext=_tick_text,
            tickfont=dict(size=14),
            x=1.01,
            thickness=14,
        ),

        showscale=True,

        # Country borders
        marker_line_width=0.3,
        marker_line_color='white',
    )

    # Curved flow arcs + arrowheads
    # -----------------------------------
    # Containers for each flow-width bucket
    bucket_lons = [[] for _ in range(N_BUCKETS)]
    bucket_lats = [[] for _ in range(N_BUCKETS)]

    # Arrowhead containers
    a_lons = [[] for _ in range(N_BUCKETS)]
    a_lats = [[] for _ in range(N_BUCKETS)]
    a_angles = [[] for _ in range(N_BUCKETS)]
    a_sizes = [[] for _ in range(N_BUCKETS)]
    a_text = [[] for _ in range(N_BUCKETS)]

    for _, row in yf.iterrows():

        # Origin and destination coordinates
        o = country_coords.get(str(row['originiso']).upper())
        a = country_coords.get(str(row['asylumiso']).upper())

        # Skip missing coordinates
        if o is None or a is None:
            continue

        # Relative flow intensity
        ratio = min(
            np.log1p(row['count']) / global_log_max_f,
            1.0
        )

        # Width bucket assignment
        bucket = min(
            int(
                max(
                    0,
                    min(
                        (
                            (np.log1p(row['count']) - np.log1p(500))
                            / (np.log1p(1_000_000) - np.log1p(500))
                        ),
                        1
                    )
                ) * N_BUCKETS
            ),
            N_BUCKETS - 1
        )


        # Generate curved path
        arc_lats, arc_lons = curved_arc(
            o[0], o[1],
            a[0], a[1]
        )

        # Store arc coordinates
        bucket_lons[bucket] += arc_lons + [None]
        bucket_lats[bucket] += arc_lats + [None]

        # Arrowhead position
        a_lons[bucket].append(a[1])
        a_lats[bucket].append(a[0])

        # Arrow orientation
        if len(arc_lats) >= 2 and len(arc_lons) >= 2:
            a_angles[bucket].append(
                calc_bearing(
                    arc_lats[-2],
                    arc_lons[-2],
                    arc_lats[-1],
                    arc_lons[-1]
                )
            )
        else:
            a_angles[bucket].append(0)

        # Arrow size proportional to flow
        # -----------------------------
        # make smaller arrows
        # a_sizes.append(4 + ratio * 8)
        # ------------------------------------------------------------------------------
        a_sizes[bucket].append(2 + ratio * 5) # minimum size of 3, maximum size of 7


        # Hover label
        a_text[bucket].append(
            f"<b>{str(row['originiso']).upper()} → {str(row['asylumiso']).upper()}</b><br>"
            f"─────────────────<br>"
            f"People moving: <b>{int(row['count']):,}</b>"
        )

    #           Line traces
    # -----------------------------------
    line_traces = []
    for b in range(N_BUCKETS):
        line_traces.append(go.Scattergeo(
            geo='geo',
            lon=bucket_lons[b] or [None],
            lat=bucket_lats[b] or [None],
            mode='lines',
            line=dict(
                width=BUCKET_WIDTHS[b],
                color=BUCKET_COLORS[b]
            ),
            showlegend=False,
            hoverinfo='skip',
        ))

    #       Arrowhead layer
    # ------------------------------------------------------------------------------
    arrow_traces = []
    for b in range(N_BUCKETS):
        arrow_traces.append(go.Scattergeo(
            geo='geo',
            lon=a_lons[b] or [None],
            lat=a_lats[b] or [None],
            mode='markers',
            marker=dict(
                symbol='arrow-up',
                size=a_sizes[b] or [0],
                color=BUCKET_COLORS[b],
                angle=a_angles[b] or [0],
                line=dict(width=0),
            ),
            text=a_text[b] or [''],
            hovertemplate='%{text}<extra></extra>',
            showlegend=False,
        ))

    #       Flow colourbar
    # ------------------------------------------------------------------------------
    flow_colorbar = go.Scattergeo(
        geo='geo',
        lon=[0],
        lat=[0],
        mode='markers',
        marker=dict(
            size=0.01,
            color=[np.log1p(500), np.log1p(1_000_000)],
            # colorscale='Purples',
            colorscale=[
                [0.00, "#f4e9ff"],
                [0.15, "#cd9dfc"],
                [0.35, "#9f47f7"],
                [0.55, "#6501c9"],
                [0.75, "#2b0048"],
                [1.00, "#000000"],
            ],
            cmin=np.log1p(500),
            cmax=np.log1p(1_000_000),
            showscale=True,
            colorbar=dict(
                title=dict(
                    text="Flow of <br>people<br>moving<br>(log scale)<br>..........",
                    font=dict(size=12),
                    side="top"
                ),
            tickvals=[
                np.log1p(500),
                np.log1p(1_000),
                np.log1p(5_000),
                np.log1p(10_000),
                np.log1p(100_000),
                np.log1p(500_000),
                np.log1p(1_000_000),
            ],

            ticktext=["500", "1k", "5k", "10k", "100k", "500k", "1M"],
                x=-0.08,
                thickness=14,
            ),
        ),
        hoverinfo='skip',
        showlegend=False,
    )

    return [choropleth] + line_traces + arrow_traces + [flow_colorbar]


# -----------------------------------
#           Initial figure
# -----------------------------------
fig = go.Figure(data=make_traces(years_list[0]))

fig.frames = [
    go.Frame(
        data=make_traces(y),
        traces=list(range(N_TRACES)),
        name=str(y)
    )
    for y in years_list
]

#       Layout control
# ----------------------------
fig.update_layout(

    title=dict(
        text='Top 100 Largest displacements worldwide per year',
        x=0.5,
        xanchor='center',
        yanchor='top',
        font=dict(color="#383838", family='helvetica', size=16)
    ),

    geo=dict(
        showframe=False,

        showcoastlines=True,
        coastlinecolor='lightgray',

        showcountries=True,
        countrycolor='white',
        countrywidth=0.7,

        showland=True,
        landcolor="#c5c5c5",

        showocean=True,
        oceancolor='#dbe9f4',

        bgcolor='#f0f0f0',
        projection_type='natural earth',
    ),
    

    # Play / pause toggle button
    updatemenus=[{
            "type": "buttons", "showactive": False,
            "y": 0.02, "x": 0.05, "xanchor": "right",
            "buttons": [
                {"label": "\u25b6 Play", "method": "animate",
                "args": [None, {"frame": {"duration": 1800}, "fromcurrent": True,
                                "transition": {"duration": 400}}]},
                {"label": "\u23f8 Pause", "method": "animate",
                "args": [[None], {"frame": {"duration": 0}, "mode": "immediate"}]},
            ],
        }],
        sliders=[{
            "active": 0,
            "steps": [
                {"args": [[str(y)], {"frame": {"duration": 1800}, "mode": "immediate",
                                    "transition": {"duration": 300}}],
                "label": str(y), "method": "animate"}
                for y in years_list
            ],
            "x": 0.1, "len": 0.85, "y": 0,
            "currentvalue": {"prefix": "Year: ", "font": {"size": 12}, "xanchor": "center"},
            "pad": {"b": 10, "t": 50},
        }],
        margin=dict(l=80, r=80, t=50, b=80),
        height=600,
    )

#            Export & show
# -----------------------------------

fig.write_html("../visualizations/Top_100_displacements_flow_map.html")

fig.show()

# 5 - A focus on three conflicts

From our mapped data, we notice the main crisis and spikes of large displacements indark red. For our story we wanted to have a look at "Where people goo?". If forced to flee, how far do people? Wioth a focus on distances, can we find patterns of behavious? is proximity a factor in displacements? or doe people prefer fleeing FAR away from war, genocide, and other horrors. As it gets easier and easier to fly accross the world, do those patterns change over time? Do people tend to move further away as time goes by?

We decided to have a deep dive at three of the most important crisis our data has to offer. For comparability we made sure to pick crisis that occur in different decades, so we can spot potential changes of behavious over time.



In [ ]:
# -----------------------------------------------------------------------------------------------
#                           Common Map code for the 3 upcoming maps 
# -----------------------------------------------------------------------------------------------

N_TRACES = 2  # only highlight + choropleth

global_max_c = 1_000_000
log_max = float(np.log1p(global_max_c))
log_min = float(np.log1p(1_000))

# colourbar ticks
_tick_raw = [1_000, 5_000, 10_000, 50_000, 100_000, 500_000, 1_000_000]
_tick_vals = [float(np.log1p(v)) for v in _tick_raw if v <= global_max_c]
_tick_text = ["1k", "5k", "10k", "50k", "100k", "500k", "1M"][:len(_tick_vals)]


#       Animation controls
# -----------------------------------

def make_animation_controls(years):

    return {
        "updatemenus": [{
            "type": "buttons",
            "showactive": False,
            "y": 0.02,
            "x": 0.05,
            "xanchor": "right",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [None, {
                        "frame": {"duration": 1800},
                        "fromcurrent": True,
                        "transition": {"duration": 400},
                    }],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [[None], {
                        "frame": {"duration": 0},
                        "mode": "immediate",
                    }],
                },
            ],
        }],

        "sliders": [{
            "active": 0,
            "steps": [
                {
                    "args": [[str(y)], {
                        "frame": {"duration": 1800},
                        "mode": "immediate",
                        "transition": {"duration": 300},
                    }],
                    "label": str(y),
                    "method": "animate",
                }
                for y in years
            ],
            "x": 0.1,
            "len": 0.85,
            "y": 0,
            "currentvalue": {
                "prefix": "Year: ",
                "font": {"size": 14},
                "xanchor": "center",
            },
            "pad": {"b": 10, "t": 50},
        }],
    }


#        Main map function
# -----------------------------------

def build_flow_map(
    origin_iso,
    start_year,
    end_year,
    highlight_color,
    title,
    output_path,
    center_lat,
    center_lon,
    zoom_scale,
):

    sub = filtered_data[
        (filtered_data["originiso"] == origin_iso)
        & (filtered_data["year"].between(start_year, end_year))
    ].copy()

    if sub.empty:
        raise ValueError(
            f"No rows found for {origin_iso} between {start_year} and {end_year}"
        )

    yearly_flows = (
        sub.groupby(["year", "originiso", "asylumiso"], as_index=False)["count"].sum()
    )

    asylum_lookup = (
        filtered_data[["asylumiso", "asylumname"]]
        .drop_duplicates()
    )

    origin_label = (
        sub["originname"].dropna().iloc[0]
        if sub["originname"].dropna().any()
        else origin_iso
    )

    years = sorted(yearly_flows["year"].unique())


    def make_traces(year):

        yf = yearly_flows[yearly_flows["year"] == year]

        dest_year = (
            yf.groupby("asylumiso", as_index=False)["count"].sum()
        )

        dest_year = dest_year.merge(
            asylum_lookup,
            on="asylumiso",
            how="left",
        )

        cd = np.column_stack([
            dest_year["asylumiso"].values,
            dest_year["asylumname"].values,
            [f"{int(x):,}" for x in dest_year["count"].values],
        ])


        # destination countries
        # ---------------------------

        choropleth = go.Choropleth(
            locations=dest_year["asylumiso"].str.upper(),

            z=np.log1p(dest_year["count"].values),

            colorscale="Reds",
            zmin=log_min,
            zmax=log_max,

            customdata=cd,

            hovertemplate=(
                "<b>%{customdata[1]}</b><br>"
                "─────────────────────────<br>"
                f"Refugees leaving {origin_label}: "
                "&nbsp;<b>%{customdata[2]}</b>"
                "<extra></extra>"
            ),

            colorbar=dict(
                title=dict(
                    text=f"Refugees<br>leaving<br>{origin_label.title()}",
                    font=dict(
                        family="Helvetica",
                        size=12,
                    ),
                ),

                tickfont=dict(
                    family="Helvetica",
                    size=12,
                ),

                tickvals=_tick_vals,
                ticktext=_tick_text,

                x=1.01,
                thickness=14,
            ),

            showscale=True,

            marker_line_width=0.3,
            marker_line_color="white",
        )


        # origin country highlight
        # ------------------------------

        highlight = go.Choropleth(
            locations=[origin_iso.upper()],

            z=[1],

            colorscale=[
                [0, highlight_color],
                [1, highlight_color],
            ],

            zmin=0,
            zmax=1,

            showscale=False,

            hovertemplate=f"<b>{origin_label.title()}</b><extra></extra>",

            marker_line_width=1.8,
            marker_line_color="white",
        )

        return [highlight, choropleth]


    # initial figure
    # -------------------

    fig = go.Figure(
        data=make_traces(years[0])
    )
    fig.frames = [
        go.Frame(
            data=make_traces(y),
            traces=[0, 1],
            name=str(y),
        )
        for y in years
    ]

    # layout
    # -----------
    fig.update_layout(
        title=dict(
            text=title,
            x=0.5,
            xanchor="center",
            font=dict(
                family="Helvetica",
                size=16,
            ),
        ),

        geo=dict(
            showframe=False,

            showcoastlines=True,
            coastlinecolor="lightgray",

            showcountries=True,
            countrycolor="white",
            countrywidth=0.7,

            showland=True,
            landcolor="#c5c5c5",

            showocean=True,
            oceancolor="#d2e5f3",

            bgcolor="#f0f0f0",

            projection_type="natural earth",

            center=dict(
                lat=center_lat,
                lon=center_lon,
            ),

            projection_scale=zoom_scale,
        ),

        hoverlabel=dict(
            bgcolor="white",
            font_size=13,
            font_family="Helvetica",
        ),

        margin=dict(
            l=0,
            r=0,
            t=50,
            b=80,
        ),

        height=600,

        **make_animation_controls(years),
    )


    # save
    # -----

    fig.write_html(output_path)

    return fig

### 5.1 Soviet-Afghan War (Dec 1979 - Feb 1989)

Though the war lasted a decade (and more), our data shows a high peak of displacements from 1980 to 1983 in particular. This leads us to believe the flow of forced displaced people is highter at the beginning of a crisis. let's go ahead and plot that to see what it looks like. 


Note on Afghanistan: the intensity and length of thos conflict makes Afghanistan somewhat of an outlier in this analysis. 

Though the conflict officially end in 1989, the instability persisted long afterward as the Taliban continued to control large parts of the country until the U.S. invasion in 2001. These decades of sustained instability produced one of the world’s largest and most prolonged refugee crises.
source: UNHCR - https://www.unhcr.org/where-we-work/countries/afghanistan?utm_source=chatgpt.com&dataset=POP&yearsMode=range&selectedYears=%5B2012%2C2026%5D&level=OPR&category=PTY&fundingSource=ALS&compareBy=%5B%22category%22%5D&levelCompare=%5B%5B%22OAFG_ABC%22%5D%5D&viewType=chart&chartType=bar&contextualDataset=BUD&tableDataView=absolute


"In fact, Afghanistan has experienced one of the world's largest refugee crises for more than two decades. Between the Soviet invasion of Afghanistan in 1979 and the present day, **one in four Afghans has been a refugee**. At the peak of the crisis in the late 1980s, there were more than six million Afghan refugees. "
source: The Forced Migration Review - https://www.fmreview.org/ruiz/

In [ ]:
# ---------------------------------------------
#     Afghanistan outflow (1979–1990)
# ---------------------------------------------

fig, ax = plt.subplots(figsize=(11, 4))

series = (
    filtered_data[filtered_data["originiso"] == "afg"]
    .groupby("year")["count"].sum()
    .reindex(range(1979, 1991), fill_value=0)
)

color = "#e0701a"

ax.fill_between(series.index, series.values, color=color, alpha=0.22)
ax.plot(series.index, series.values, color=color, linewidth=2)

# peak year
peak_year = series.idxmax()
peak_val = series.max()

ax.axvline(
    peak_year,
    color=color,
    linestyle="--",
    linewidth=1.2,
    alpha=0.7,
)

ax.annotate(
    f"{peak_year}\n{int(peak_val):,}",
    xy=(peak_year, peak_val),
    xytext=(20, 20),
    textcoords="offset points",
    fontsize=10,
    fontweight="bold",
    color=color,
    va="top",
)

# centered title
ax.set_title(
    "Afghanistan refugee outflow (1979-1990)",
    fontsize=13,
    pad=25,
    fontfamily="Helvetica",
)

ax.set_xlabel("Year", labelpad=15, fontfamily="Helvetica", fontsize=12)
ax.set_ylabel("People displaced", labelpad=15, fontfamily="Helvetica", fontsize=12)

# show every year
ax.set_xticks(range(1979, 1991))

ax.grid(alpha=0.25)

for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)

ax.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda x, _: (
            f"{int(x / 1e6)}M"
            if x >= 1e6
            else f"{int(x / 1e3)}k"
        )
    )
)

plt.tight_layout()

plt.savefig(
    "../visualizations/afghanistan_outflow_1979_1990.png",
    dpi=150,
    bbox_inches="tight",
)

plt.show()

The soviets started their occupation (between 1980 - 1085) with a "scorched earth campaign", which aimed to cut any support the mujahideen (religious Afghan freedom fighters) might have in the region. This implied the destrucyion of crops, irrication systems, and villages and affected local population terribly. By the end of 1981, the widespread destruction of civilian infrastructure further intensified displacement and cross-border asylum movements.

Source: This article written my HIstory professor Edward B. Westermann
https://journals.lib.unb.ca/index.php/jcs/article/view/4356/5011

"Soviet attempts to intimidate the civil population involved a joint air-land effort. The intensive bombardment of villages [...] erved as the prelude for the entry of mechanized and armored forces into the area. These forces then proceeded to conduct a "scorched earth" campaign by destroying the local dwellings, food supplies, crops in the field, irrigation systems, livestock and wells. One Swedish official, after visiting several villages destroyed by the Soviets noted, "Russian soldiers shot at anything alive in six villages -- people, hens, donkeys - and then they plundered what remained of value."16 These Soviet operations aimed at driving the villagers out of these areas in an effort to create a cordon sanitaire in which the insurgents would find no support. 

In [ ]:
#                       LONG SHIT 
# ------------------------------------------------------------------------




# -----------------------------------------------------------------------------------------------
#                           Common Map code for the 3 upcoming maps
# -----------------------------------------------------------------------------------------------

N_TRACES = 3  # highlight + choropleth + USSR overlay

global_max_c = 1_000_000
log_max = float(np.log1p(global_max_c))
log_min = float(np.log1p(1_000))

# colourbar ticks
_tick_raw = [1_000, 5_000, 10_000, 50_000, 100_000, 500_000, 1_000_000]
_tick_vals = [float(np.log1p(v)) for v in _tick_raw if v <= global_max_c]
_tick_text = ["1k", "5k", "10k", "50k", "100k", "500k", "1M"][:len(_tick_vals)]


#       Animation controls
# -----------------------------------

def make_animation_controls(years):

    return {
        "updatemenus": [{
            "type": "buttons",
            "showactive": False,
            "y": 0.02,
            "x": 0.05,
            "xanchor": "right",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [None, {
                        "frame": {"duration": 1800},
                        "fromcurrent": True,
                        "transition": {"duration": 400},
                    }],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [[None], {
                        "frame": {"duration": 0},
                        "mode": "immediate",
                    }],
                },
            ],
        }],

        "sliders": [{
            "active": 0,
            "steps": [
                {
                    "args": [[str(y)], {
                        "frame": {"duration": 1800},
                        "mode": "immediate",
                        "transition": {"duration": 300},
                    }],
                    "label": str(y),
                    "method": "animate",
                }
                for y in years
            ],
            "x": 0.1,
            "len": 0.85,
            "y": 0,
            "currentvalue": {
                "prefix": "Year: ",
                "font": {"size": 14},
                "xanchor": "center",
            },
            "pad": {"b": 10, "t": 50},
        }],
    }


#       USSR historical overlay
# -----------------------------------

def add_ussr_1980_overlay(fig):

    ussr_geojson_url = (
        "https://raw.githubusercontent.com/ioggstream/"
        "europe-historical-geojson/main/urss.geojson"
    )

    ussr_geojson = requests.get(ussr_geojson_url).json()

    # dissolve all USSR parts into one geometry
    ussr_geometry = unary_union([
        shape(feature["geometry"])
        for feature in ussr_geojson["features"]
    ])

    ussr_geojson = {
        "type": "FeatureCollection",
        "features": [
            {
                "type": "Feature",
                "id": "USSR_1980",
                "properties": {
                    "name": "USSR boundary, circa 1980",
                },
                "geometry": mapping(ussr_geometry),
            }
        ],
    }

    fig.add_trace(
        go.Choropleth(
            geojson=ussr_geojson,
            locations=["USSR_1980"],
            z=[1],

            colorscale=[
                [0, "rgba(197, 197, 197, 1)"],
                [1, "rgba(197, 197, 197, 1)"],
            ],

            marker_line_color="white",
            marker_line_width=0.5,

            showscale=False,
            showlegend=False,

            hovertemplate="<b>USSR boundary, circa 1980</b><extra></extra>",
            name="USSR 1980",
        )
    )

    return fig


#        Main map function
# -----------------------------------

def build_flow_map(
    origin_iso,
    start_year,
    end_year,
    highlight_color,
    title,
    output_path,
    center_lat,
    center_lon,
    zoom_scale,
    show_ussr_1980_overlay=False,
):

    sub = filtered_data[
        (filtered_data["originiso"] == origin_iso)
        & (filtered_data["year"].between(start_year, end_year))
    ].copy()

    if sub.empty:
        raise ValueError(
            f"No rows found for {origin_iso} between {start_year} and {end_year}"
        )

    yearly_flows = (
        sub.groupby(["year", "originiso", "asylumiso"], as_index=False)["count"].sum()
    )

    asylum_lookup = (
        filtered_data[["asylumiso", "asylumname"]]
        .drop_duplicates()
    )

    origin_label = (
        sub["originname"].dropna().iloc[0]
        if sub["originname"].dropna().any()
        else origin_iso
    )

    years = sorted(yearly_flows["year"].unique())


    def make_traces(year):

        yf = yearly_flows[yearly_flows["year"] == year]

        dest_year = (
            yf.groupby("asylumiso", as_index=False)["count"].sum()
        )

        dest_year = dest_year.merge(
            asylum_lookup,
            on="asylumiso",
            how="left",
        )

        cd = np.column_stack([
            dest_year["asylumiso"].values,
            dest_year["asylumname"].values,
            [f"{int(x):,}" for x in dest_year["count"].values],
        ])


        # destination countries
        # ---------------------------

        choropleth = go.Choropleth(
            locations=dest_year["asylumiso"].str.upper(),

            z=np.log1p(dest_year["count"].values),

            colorscale="Reds",
            zmin=log_min,
            zmax=log_max,

            customdata=cd,

            hovertemplate=(
                "<b>%{customdata[1]}</b><br>"
                "─────────────────────────<br>"
                f"Refugees leaving {origin_label}: "
                "&nbsp;<b>%{customdata[2]}</b>"
                "<extra></extra>"
            ),

            colorbar=dict(
                title=dict(
                    text=f"Refugees<br>leaving<br>{origin_label.title()}",
                    font=dict(
                        family="Helvetica",
                        size=12,
                    ),
                ),

                tickfont=dict(
                    family="Helvetica",
                    size=12,
                ),

                tickvals=_tick_vals,
                ticktext=_tick_text,

                x=1.01,
                thickness=14,
            ),

            showscale=True,

            marker_line_width=0.3,
            marker_line_color="white",
        )


        # origin country highlight
        # ------------------------------

        highlight = go.Choropleth(
            locations=[origin_iso.upper()],

            z=[1],

            colorscale=[
                [0, highlight_color],
                [1, highlight_color],
            ],

            zmin=0,
            zmax=1,

            showscale=False,

            hovertemplate=f"<b>{origin_label.title()}</b><extra></extra>",

            marker_line_width=1.8,
            marker_line_color="white",
        )

        return [highlight, choropleth]


    # initial figure
    # -------------------

    fig = go.Figure(
        data=make_traces(years[0])
    )

    fig.frames = [
        go.Frame(
            data=make_traces(y),
            traces=[0, 1],
            name=str(y),
        )
        for y in years
    ]


    # layout
    # -----------

    fig.update_layout(
        title=dict(
            text=title,
            x=0.5,
            xanchor="center",
            font=dict(
                family="Helvetica",
                size=16,
            ),
        ),

        geo=dict(
            showframe=False,

            showcoastlines=True,
            coastlinecolor="lightgray",

            showcountries=True,
            countrycolor="white",
            countrywidth=0.7,

            showland=True,
            landcolor="#c5c5c5",

            showocean=True,
            oceancolor="#d2e5f3",

            bgcolor="#f0f0f0",

            projection_type="natural earth",

            center=dict(
                lat=center_lat,
                lon=center_lon,
            ),

            projection_scale=zoom_scale,
        ),

        hoverlabel=dict(
            bgcolor="white",
            font_size=13,
            font_family="Helvetica",
        ),

        margin=dict(
            l=0,
            r=0,
            t=50,
            b=80,
        ),

        height=600,

        **make_animation_controls(years),
    )


    # optional historical USSR overlay
    # -----------------------------------

    if show_ussr_1980_overlay:
        add_ussr_1980_overlay(fig)


    # save
    # -----

    fig.write_html(output_path)

    return fig

In [ ]:

#                       SHORTER SHIT
#  ----------------------------------------------------------------------------

# Generate maps for three conflict scenarios

afghanistan_fig = build_flow_map(
    "afg",
    1980,
    1983,
    "rgba(241, 196, 15, 0.88)",
    "Afghan refugees and their destinations between 1980 and 1983",
    "../visualizations/afghanistan_flows_1980_1983.html",

    # map centered on Afghanistan
    center_lat=33,
    center_lon=45,
    zoom_scale=2.5,

    show_ussr_1980_overlay=True,
)

afghanistan_fig.show()


In [ ]:
# Generate maps for three conflict scenarios
afghanistan_fig = build_flow_map(
    "afg",
    1980,
    1983,
    "rgba(241, 196, 15, 0.88)",
    "Afghan refugees and their destination between 1980 and 1983",
    "../visualizations/afghanistan_flows_1980_1983.html",
    # mao centered on Afghanistan
    center_lat=33,
    center_lon=45,
    zoom_scale=2.5,
)

afghanistan_fig.show()

In [ ]:
# ---------------------------------------------------------
#      Refugee destinations by year (1980–1983)
# ---------------------------------------------------------

table = (
    filtered_data[
        (filtered_data["originiso"] == "afg") &
        (filtered_data["year"].between(1980, 1983))
    ]
    .groupby(["asylumname", "year"])["count"]
    .sum()
    .unstack(fill_value=0)
)

# sort by total refugees received
table = table.loc[
    table.sum(axis=1).sort_values(ascending=False).index
]

# format numbers nicely
def pretty(x):

    if x >= 1_000_000:
        return f"{x/1_000_000:.1f}M"

    elif x >= 1_000:
        return f"{x/1_000:.0f}k"

    return str(int(x))

table_display = table.map(pretty)

table_display

#### For our story - where do people go?

- We immediately notice Iran in dark red taking in a whooping 1.4 million people in 1981, followed closely by Pakistan with close to a million refugees. those are two adjascent countries.

- But what about the other adjacent countries north of the country: Turkmenistan, Uzbekistan, and Tajikistan? Why are they marked in grey? Did they not take in any refugees? ...Not quite.  
You may remember at the beginning of the analysis that we mentioned the maps did not take into consideration the creation of new countries over time. A Folium map usually uses current country borders, because it relies on modern basemap tiles or modern GeoJSON/shapefiles. Back in 1980 ,The three aforementioned countries were still part of the USSR bloc and therefore were considered part of the direct attacking force, explaining more clearly why no Afghan would seek refuge up north. 

- Iran from 1.4M in 1981 to 0 in 1982?
Why do some countries take in massive numbers one year and then 0 the following year? Iran 1.4M in 1981 and then 0 in 1982? Policy changes? Did the borders suddenly close? Two things:
    - **Status name:** Upon closer inspection, it would rather seem that this is more a question of “status” than an actual halt in displacement flows. Though the borders were indeed opnened during this period, displaced people were not labeled as "refugees" and could therefore be unregisttered in our UNHCR dataset. So our data may show 0 while in reality hundreds were still physically crossing into Iran.

    - **UNHCR reconstruction of numbers retrospectively:** It may have been hard to keep track of the exact flux taking place each year of the conflict, and it could be that some records were incomplete, estimated retrospectively, or merged into a single year. This is something that can occur with older displacement data. We therefore suspect that something may have gone wrong in the way the data was recorded or reconstructed, especially since we know for a fact that the borders remained open during this period.

"Until 1992, the Iranian government allowed many Afghans to register as “involuntary migrants” (mohajerin), gain automaticresidency in Iran, and enjoy benefits such as basic healthcare and work permits. Iranian officials effectively treated these registered Afghans as refugees, though that designation was not officially used by the Iranian government"
Source: Human Rights Watch
https://www.hrw.org/reports/iran1113_ForUpload_0.pdf?utm_source=chatgpt.com


- As seen in our preliminary study, we have few data points around this decade. (backs up the abovementioned)

### 5.2 - Rwanda Genocide (1994 -1996)

In [ ]:
rwanda_fig = build_flow_map(
    "rwa",
    1994,
    1996,
    "rgba(241, 196, 15, 0.88)",
    "Rwanda refugee flows 1994-1996",
    "../visualizations/rwanda_flows_1994_1996.html",
    # map centeres in Rwanda
    center_lat=-1,
    center_lon=30,
    zoom_scale=5,
)
rwanda_fig.show()

### 5.3 - Russia's invation of Ukraine (2022 - 2025, when the data stops)

In [ ]:
ukraine_fig = build_flow_map(
    "ukr",
    2022,
    2025,
    "rgba(241, 196, 15, 0.88)",
    "Ukraine refugee flows 2022-2025",
    "../visualizations/ukraine_flows_2022_2025.html",
    # map centered on Ukraine
    center_lat=49,
    center_lon=32,
    zoom_scale=3.0,
)
ukraine_fig.show()

# 6 - Comparing the three crisis

In [ ]:
# ---------------------------------------------
#         Plot to compare the outflow
# ---------------------------------------------
CRISES = [
    ("afg", "Afghanistan", 1980, "#e0701a"),
    ("rwa", "Rwanda",      1994, "#7ad723"),
    ("ukr", "Ukraine",     2022, "#a434c6"),
]

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True, sharey=True)

for ax, (iso, name, peak, color) in zip(axes, CRISES):
    series = (
        filtered_data[filtered_data["originiso"] == iso]
        .groupby("year")["count"].sum()
        .reindex(range(1962, 2026), fill_value=0)
    )

    ax.fill_between(series.index, series.values, color=color, alpha=0.22)
    ax.plot(series.index, series.values, color=color, linewidth=2)

    ax.axvline(peak, color=color, linestyle="--", linewidth=1.2, alpha=0.7)

    peak_val = series.loc[peak] if peak in series.index else 0
    ax.annotate(
        f"{peak}\n{int(peak_val):,}",
        xy=(peak, peak_val),
        xytext=(20, 20),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold",
        color=color,
        va="top",
    )

    ax.set_title(f"{name} ({iso.upper()})", loc="left", fontsize=12, pad=15, fontfamily="Helvetica")
    ax.set_ylabel("People displaced")
    ax.grid(alpha=0.25)

    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)

    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(
            lambda x, _: f"{int(x / 1e6)}M" if x >= 1e6 else f"{int(x / 1e3)}k"
        )
    )

axes[-1].set_xlabel("Year")

fig.suptitle(
    "Annual outflow from three crisis countries",
    fontsize=14,
    fontfamily="Helvetica",
    fontweight="bold",
    y=1.0,
)

fig.tight_layout()
fig.subplots_adjust(hspace=0.5)

plt.savefig("../visualizations/crisis_outflows.png", dpi=150, bbox_inches="tight")
plt.show()

# Draft

In [ ]:
# -----------------------------------------------------------------
#                       Things to talk about ? 
# -----------------------------------------------------------------

"""

 EXPLORATION OF THE DATA 
            <br>
            whata re the top 3 events in our data that caused the largest displacements?
            <br>
            (this is a good spot for Loke's chart).
            0 - filtering: only keep "significant" displacements, removeless than 100 people entries +  displacement from 10.000 people per year = min threshold, 
            or use median if number of displaced is larger than 10K
            <br>
            2 - in sheer numbers ...what are the largest displacements recorded in our data 
            3 - taking into consideration the length of the conflict/crisis. 1 million people moving over 20 years doesn't have the same impart as 
            1 million people moving over 1 year. so we can compute a "ratio" of "total displaced" VS "annual displaced"
            <img src="TishDuc.github.io/Displacements_Final_Project/Draft/Top_10.png"/>
            <p class="caption">
            Top 10 events in the modern era that caused the largest displacements. Computed taking into consideration the length of the crisis, 
            to show the "intensity" of the displacements.
            </p>
            <br>
            OUR DATA: UNHCR data on displacements around the world, from 1962 to 2025 + other data GDP? country population (to compute ratios)? 
        </p>
        </section>


        </section>
        <section class = "text"></section>

        DRAFT OF THE STORY:
            <br>
            <br>
            - Ratio thing: Natural disasters/conflicts/war have different lengths. find a way to express the "amount" of displcement 
            taking that into consideration. 1 million people moving over 20 years doesn't have the same impart as 
            1 million people moving over 1 year. 
            Total displaced VS annual displaced
            <br>
            <br>
            - Filtering the data: 
            crisis = 10.000 peple displaced in 1 year - OR - use a "percentage/median" of the total population of the country (example of Afghanistan)
            removed "tiny" values, only kept movements of more than 100 people 
            <br>
            <br>
            - Sometimes, lots of poeple move from A to B (single country), other times people move from B to B, C, D,..(several countries)
            why? is this something to look into?
            <br>
            <br>
            - Early data: Some large conflicts from the 70's80's (Afganistan/Ethiopia) make it in out "Top" 20 in sheer number of displaced.
            Yet data of that period is so sparce/so rare at the time. Does that mean that the actual number of displaced was HUUUUUGE?
            <br>
            <br>
            - Displacements types: Asylum seekers, refugees....
            Do we look into the different status of displaced people? 
            <br>
            <br>
            - Suspiciously round numbers: in the data ... were there exactly 20,000 people moving from Angola to Congo in 1962?
            looks suspicious to me?
            <br>
            <br>
            - Why do they "choose" the country the go to? 
            is it because of language? culture/religoin? proximity? economic opportunities? easier migration policies?
            <br>
            <br>
            LIMITS OF OUR DATASET: 
            <br>
            <br>
            - Only tracks INTERNATIONAL movements... so it' snot adequate for "natural disasters", or local population movements
            not adequete for tsunami in 2006 in indonesia: too many different countries around the Indian Ocean.
            partly Adequate for Tchernobyl in 1986?...ish (people moving from Ukraine to Russia)? but not tracting russian people moving elsewhere to Russia.
            <br>
            <br>
            - Displacements are recorded per year: so we can-t look at the "immediate" impact of a crisis, but rather the "long-term" impact. 
            and we can't look at super short yet intense crisis.
            <br>
            <br>
            limited data from 1960 to ...1990...2000
            <br>
            <br>
            Keeping in mind the numbers we have are only the "ocfficially" recorded displacements... there are probably many more.
            <br>
            <br>
            - "unknown" or "stateless" displacements: 
            Do we count "unkown" as missing data?
            Or are they called "unknown" because origin countries were renamed/created/disappeared OR are those the precisely 
            the "stateless" entries? foe example we soo lots of "stateless" displacements entering Europe in 1989... ex-Yougoslavia?
            In some cases (Uganda) it be that people they wish to remain unknown for their own safety...for political/ethnicity/religious reasons 
         
            

"""